# TASK 1 : ASR[Arabic Speech Recognition]

#### Try out more samples of the dataset when downloaded and calculate the WER and CER 

In [2]:
# ============================================================
# Prepare df for Task 1 ASR Evaluation — Current Downloaded Speech Dataset
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
AUDIO_DIR = DATA_DIR / "audio"
EVAL_DIR = PROJECT_ROOT / "outputs_clean_final" / "evaluation"

AUDIO_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

COMMON_VOICE_VALIDATION_FILE = (
    DATA_DIR
    / "common_voice_18_arabic"
    / "data"
    / "validation-00000-of-00001.parquet"
)

if not COMMON_VOICE_VALIDATION_FILE.exists():
    raise FileNotFoundError(f"Missing speech dataset file: {COMMON_VOICE_VALIDATION_FILE}")

speech_df = pd.read_parquet(COMMON_VOICE_VALIDATION_FILE)

print("Speech dataset loaded:")
print("Shape:", speech_df.shape)
print("Columns:", speech_df.columns.tolist())

TASK1_AUDIO_DIR = AUDIO_DIR / "task1_asr_samples"
TASK1_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

N_TASK1_SAMPLES = 5

task1_rows = []

for i in range(N_TASK1_SAMPLES):
    sample = speech_df.iloc[i]

    audio_data = sample["audio"]
    reference_sentence = str(sample["sentence"]).strip()

    if not isinstance(audio_data, dict):
        continue

    audio_bytes = audio_data.get("bytes", None)

    if audio_bytes is None:
        continue

    audio_path = TASK1_AUDIO_DIR / f"task1_sample_{i}.mp3"

    with open(audio_path, "wb") as f:
        f.write(audio_bytes)

    task1_rows.append({
        "audio_path": str(audio_path),
        "sentence": reference_sentence,
        "dataset_source": "common_voice_18_arabic"
    })

df = pd.DataFrame(task1_rows)

print("\nTask 1 ASR dataframe created:")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df)

Speech dataset loaded:
Shape: (10471, 12)
Columns: ['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment', 'variant']

Task 1 ASR dataframe created:
Shape: (5, 3)
Columns: ['audio_path', 'sentence', 'dataset_source']


,audio_path,sentence,dataset_source
0,d:\sara\Project 2 NLP\data\audio\task1_asr_sam...,كل ما عليك هو إمضاء إسمك هنا,common_voice_18_arabic
1,d:\sara\Project 2 NLP\data\audio\task1_asr_sam...,.الى متى سنقف مكتوفى الايدى,common_voice_18_arabic
2,d:\sara\Project 2 NLP\data\audio\task1_asr_sam...,هذا هو الخيار الوحيد.,common_voice_18_arabic
3,d:\sara\Project 2 NLP\data\audio\task1_asr_sam...,استمرت الحرب قرابة السنتين.,common_voice_18_arabic
4,d:\sara\Project 2 NLP\data\audio\task1_asr_sam...,.تزداد الحوادث يوماً بعد يوم,common_voice_18_arabic


In [3]:
# ============================================================
# Task 1 — Arabic ASR Evaluation with Whisper
# Fixes long audio error using return_timestamps=True
# ============================================================

import os
import re
import torch
import pandas as pd
from pathlib import Path
from transformers import pipeline
from jiwer import wer, cer

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

ASR_MODEL_NAME = "openai/whisper-medium"
ASR_DEVICE = 0 if torch.cuda.is_available() else -1

AUDIO_PATH_COL = "audio_path"
REFERENCE_COL = "sentence"

N_EVAL_SAMPLES = 5

print("ASR model:", ASR_MODEL_NAME)
print("ASR device:", "GPU" if ASR_DEVICE == 0 else "CPU")

# ------------------------------------------------------------
# 2. Arabic normalization
# ------------------------------------------------------------

def normalize_arabic_for_wer(text):
    text = str(text)

    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)

    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)
    text = re.sub("ة", "ه", text)

    text = re.sub("ـ", "", text)

    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

# ------------------------------------------------------------
# 3. Check df
# ------------------------------------------------------------

if "df" not in globals():
    raise NameError(
        "df is not defined. Run the previous prepare cell first. "
        "It should create df with columns: audio_path and sentence."
    )

required_cols = [AUDIO_PATH_COL, REFERENCE_COL]
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing columns in df: {missing_cols}. Current columns: {df.columns.tolist()}")

print("Evaluation dataframe shape:", df.shape)
print("Evaluation dataframe columns:", df.columns.tolist())

# ------------------------------------------------------------
# 4. Load Whisper ASR pipeline
# ------------------------------------------------------------

asr_pipeline = pipeline(
    task="automatic-speech-recognition",
    model=ASR_MODEL_NAME,
    device=ASR_DEVICE,
    chunk_length_s=30,
    stride_length_s=5
)

print("ASR pipeline loaded successfully.")

# ------------------------------------------------------------
# 5. Safe transcription function
# ------------------------------------------------------------

def transcribe_audio_for_eval(audio_path):
    result = asr_pipeline(
        str(audio_path),
        return_timestamps=True,
        generate_kwargs={
            "language": "arabic",
            "task": "transcribe",
            "num_beams": 5
        }
    )

    predicted_text = result.get("text", "").strip()
    return predicted_text
# ------------------------------------------------------------
# 6. Run ASR evaluation
# ------------------------------------------------------------

print("Starting Task 1 Evaluation...")

eval_rows = []

eval_df = df.head(N_EVAL_SAMPLES).copy()

for i, row in eval_df.iterrows():
    try:
        audio_path = row[AUDIO_PATH_COL]
        reference_text = str(row[REFERENCE_COL]).strip()

        print(f"\nProcessing sample {len(eval_rows)+1}/{len(eval_df)}")
        print("Audio:", audio_path)

        predicted_text = transcribe_audio_for_eval(audio_path)

        reference_norm = normalize_arabic_for_wer(reference_text)
        predicted_norm = normalize_arabic_for_wer(predicted_text)

        sample_wer = wer(reference_norm, predicted_norm)
        sample_cer = cer(reference_norm, predicted_norm)

        eval_rows.append({
            "sample_id": len(eval_rows),
            "audio_path": str(audio_path),
            "reference_text": reference_text,
            "predicted_text": predicted_text,
            "reference_norm": reference_norm,
            "predicted_norm": predicted_norm,
            "wer": sample_wer,
            "cer": sample_cer,
            "wer_percent": sample_wer * 100,
            "cer_percent": sample_cer * 100
        })

        print("Reference:", reference_text)
        print("Predicted:", predicted_text)
        print(f"WER: {sample_wer * 100:.2f}%")
        print(f"CER: {sample_cer * 100:.2f}%")

    except Exception as e:
        print(f"Error on sample {len(eval_rows)+1}: {e}")

# ------------------------------------------------------------
# 7. Results
# ------------------------------------------------------------

asr_eval_df = pd.DataFrame(eval_rows)

if len(asr_eval_df) == 0:
    raise ValueError("No samples were evaluated successfully. Check audio paths and column names.")

asr_summary_df = pd.DataFrame([{
    "model_name": ASR_MODEL_NAME,
    "num_samples": len(asr_eval_df),
    "mean_wer": asr_eval_df["wer"].mean(),
    "mean_cer": asr_eval_df["cer"].mean(),
    "mean_wer_percent": asr_eval_df["wer_percent"].mean(),
    "mean_cer_percent": asr_eval_df["cer_percent"].mean()
}])

display(asr_eval_df[[
    "sample_id",
    "wer_percent",
    "cer_percent",
    "reference_text",
    "predicted_text"
]])

display(asr_summary_df)

# ------------------------------------------------------------
# 8. Save outputs
# ------------------------------------------------------------

asr_eval_path = EVAL_DIR / "task1_asr_eval.csv"
asr_summary_path = EVAL_DIR / "task1_asr_eval_summary.csv"

asr_eval_df.to_csv(asr_eval_path, index=False, encoding="utf-8-sig")
asr_summary_df.to_csv(asr_summary_path, index=False, encoding="utf-8-sig")

print("Saved ASR evaluation to:", asr_eval_path)
print("Saved ASR summary to:", asr_summary_path)

d:\sara\Project 2 NLP\arabic_rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ASR model: openai/whisper-medium
ASR device: GPU
Evaluation dataframe shape: (5, 3)
Evaluation dataframe columns: ['audio_path', 'sentence', 'dataset_source']


Loading weights: 100%|██████████| 947/947 [00:00<00:00, 6121.68it/s]
[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
[transformers] Passing `generation_config` together with generation-related arguments=({'num_beams'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


ASR pipeline loaded successfully.
Starting Task 1 Evaluation...

Processing sample 1/5
Audio: d:\sara\Project 2 NLP\data\audio\task1_asr_samples\task1_sample_0.mp3


[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
[

Reference: كل ما عليك هو إمضاء إسمك هنا
Predicted: كل ما عليك هو امضاء اسمك هنا
WER: 0.00%
CER: 0.00%

Processing sample 2/5
Audio: d:\sara\Project 2 NLP\data\audio\task1_asr_samples\task1_sample_1.mp3
Reference: .الى متى سنقف مكتوفى الايدى
Predicted: إلى متى سنقف مكتوفي الأيدي؟
WER: 20.00%
CER: 3.85%

Processing sample 3/5
Audio: d:\sara\Project 2 NLP\data\audio\task1_asr_samples\task1_sample_2.mp3
Reference: هذا هو الخيار الوحيد.
Predicted: هذا هو الخيار الوحيد
WER: 0.00%
CER: 0.00%

Processing sample 4/5
Audio: d:\sara\Project 2 NLP\data\audio\task1_asr_samples\task1_sample_3.mp3
Reference: استمرت الحرب قرابة السنتين.
Predicted: استمرت الحرب قرابة السنتين
WER: 0.00%
CER: 0.00%

Processing sample 5/5
Audio: d:\sara\Project 2 NLP\data\audio\task1_asr_samples\task1_sample_4.mp3
Reference: .تزداد الحوادث يوماً بعد يوم
Predicted: تزداد الحوادث يوما بعد يوم
WER: 0.00%
CER: 0.00%


,sample_id,wer_percent,cer_percent,reference_text,predicted_text
0,0,0.0,0.000000,كل ما عليك هو إمضاء إسمك هنا,كل ما عليك هو امضاء اسمك هنا
1,1,20.0,3.846154,.الى متى سنقف مكتوفى الايدى,إلى متى سنقف مكتوفي الأيدي؟
2,2,0.0,0.000000,هذا هو الخيار الوحيد.,هذا هو الخيار الوحيد
3,3,0.0,0.000000,استمرت الحرب قرابة السنتين.,استمرت الحرب قرابة السنتين
4,4,0.0,0.000000,.تزداد الحوادث يوماً بعد يوم,تزداد الحوادث يوما بعد يوم


,model_name,num_samples,mean_wer,mean_cer,mean_wer_percent,mean_cer_percent
0,openai/whisper-medium,5,0.04,0.007692,4.0,0.769231


Saved ASR evaluation to: d:\sara\Project 2 NLP\outputs_clean_final\evaluation\task1_asr_eval.csv
Saved ASR summary to: d:\sara\Project 2 NLP\outputs_clean_final\evaluation\task1_asr_eval_summary.csv


# TASK 2: Summarization

In [5]:
import torch
import re
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
from rouge_score import rouge_scorer

# --- 1. SETUP & MODEL LOADING ---
SUMM_MODEL_NAME = "csebuetnlp/mT5_multilingual_XLSum"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading Model: {SUMM_MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(SUMM_MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(SUMM_MODEL_NAME).to(device)

# --- 2. DATASET LOADING ---
print("Loading XLSum Arabic dataset...")
summ_dataset = load_dataset("csebuetnlp/xlsum", "arabic", streaming=False, trust_remote_code=True)
# We use 20 samples to ensure the ROUGE score is stable and statistically significant
test_samples = summ_dataset['test'].select(range(20))

# --- 3. CLEANING & GENERATION FUNCTIONS ---
def clean_arabic_report(text):
    text = str(text)
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text) # Remove diacritics
    text = text.replace("ة", "ه").replace("ى", "ي").replace("إ", "ا").replace("أ", "ا").replace("آ", "ا")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def generate_summary(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    output_tokens = model.generate(
        inputs["input_ids"],
        max_length=100,
        num_beams=4,
        no_repeat_ngram_size=3,
        early_stopping=True
    )
    return tokenizer.decode(output_tokens[0], skip_special_tokens=True)

# --- 4. EXECUTION ---
print("Generating summaries for evaluation...")
preds = []
refs = []

for i, sample in enumerate(test_samples):
    raw_pred = generate_summary(sample['text'])
    
    # We clean both to ensure minor spelling variations don't hurt the score
    preds.append(clean_arabic_report(raw_pred))
    refs.append(clean_arabic_report(sample['summary']))
    
    if (i+1) % 5 == 0:
        print(f"Progress: {i+1}/20")

# --- 5. FINAL SCORING (Your Best Method) ---
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], tokenizer=tokenizer)

all_scores = []
for p, r in zip(preds, refs):
    score = scorer.score(r, p)
    all_scores.append({
        'rouge1': score['rouge1'].fmeasure,
        'rouge2': score['rouge2'].fmeasure,
        'rougeL': score['rougeL'].fmeasure
    })

results_df = pd.DataFrame(all_scores)
final_metrics = results_df.mean() * 100

summary_table = pd.DataFrame({
    "Metric": ["ROUGE-1", "ROUGE-2", "ROUGE-L"],
    "Score (%)": [final_metrics['rouge1'], final_metrics['rouge2'], final_metrics['rougeL']]
})

print("\n--- Final Validated Task 2 Metrics ---")
display(summary_table)

Loading Model: csebuetnlp/mT5_multilingual_XLSum...


Loading weights: 100%|██████████| 284/284 [00:00<00:00, 613063.48it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loading XLSum Arabic dataset...
Generating summaries for evaluation...
Progress: 5/20
Progress: 10/20
Progress: 15/20
Progress: 20/20

--- Final Validated Task 2 Metrics ---


,Metric,Score (%)
0,ROUGE-1,42.924477
1,ROUGE-2,23.293367
2,ROUGE-L,33.606097


### Task 3: Semantic Search (ARCD)

In [6]:
# ============================================================
# Task 3 — Load ARCD Dataset for Arabic Semantic Search
# ============================================================

from datasets import load_dataset
import pandas as pd

print("Loading ARCD dataset...")

# Try loading ARCD from Hugging Face
arcd_dataset = load_dataset("hsseinmz/arcd")

print(arcd_dataset)

# Use validation if available, otherwise train
if "validation" in arcd_dataset:
    arcd_split = arcd_dataset["validation"]
else:
    arcd_split = arcd_dataset["train"]

arcd_df = arcd_split.to_pandas()

print("ARCD shape:", arcd_df.shape)
print("Columns:", arcd_df.columns.tolist())

display(arcd_df.head())

Loading ARCD dataset...
DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 693
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 702
    })
})
ARCD shape: (702, 5)
Columns: ['id', 'title', 'context', 'question', 'answers']


,id,title,context,question,answers
0,621723207492,حمزة بن عبد المطلب,حمزة بن عبد المطلب الهاشمي القرشي صحابي من صحا...,من هو حمزة بن عبد المطلب؟,{'text': ['صحابي من صحابة رسول الإسلام محمد، و...
1,189105393656,حمزة بن عبد المطلب,حمزة بن عبد المطلب الهاشمي القرشي صحابي من صحا...,بما وصفه رسول الله؟,"{'text': ['وَخَيْرُ أَعْمَامِي'], 'answer_star..."
2,662616978980,حمزة بن عبد المطلب,حمزة بن عبد المطلب الهاشمي القرشي صحابي من صحا...,بما وصف رسول الله على ؟,"{'text': ['«خَيْرُ إِخْوَتِي عَلِيٌّ،'], 'answ..."
3,50146585922,حمزة بن عبد المطلب,أسلم حمزة في السنة الثانية من بعثة النبي محمد،...,متى اسلم حمزة؟,{'text': ['في السنة الثانية من بعثة النبي محمد...
4,259563807541,حمزة بن عبد المطلب,أسلم حمزة في السنة الثانية من بعثة النبي محمد،...,و ماذا فعل فى غزوة بدر؟,{'text': ['وقَتَلَ فيها شيبة بن ربيعة مبارزةً،...


In [7]:
# ============================================================
# Inspect ARCD Columns
# ============================================================

print("Columns:", arcd_df.columns.tolist())

for col in arcd_df.columns:
    print("\n" + "=" * 80)
    print("Column:", col)
    print("First value:")
    print(arcd_df[col].iloc[0])

Columns: ['id', 'title', 'context', 'question', 'answers']

Column: id
First value:
621723207492

Column: title
First value:
حمزة بن عبد المطلب

Column: context
First value:
حمزة بن عبد المطلب الهاشمي القرشي صحابي من صحابة رسول الإسلام محمد، وعمُّه وأخوه من الرضاعة وأحد وزرائه الأربعة عشر، وهو خير أعمامه لقوله: «خَيْرُ إِخْوَتِي عَلِيٌّ، وَخَيْرُ أَعْمَامِي حَمْزَةُ رَضِيَ اللَّهُ عَنْهُمَا».

Column: question
First value:
من هو حمزة بن عبد المطلب؟

Column: answers
First value:
{'text': array(['صحابي من صحابة رسول الإسلام محمد، وعمُّه وأخوه من الرضاعة وأحد وزرائه الأربعة عشر،'],
      dtype=object), 'answer_start': array([34], dtype=int32)}


In [8]:
# ============================================================
# Cell 3 — Prepare ARCD Question-Context Pairs with Title
# ============================================================

question_col = "question"
context_col = "context"
title_col = "title"

search_df = arcd_df[[title_col, question_col, context_col]].dropna().copy()

search_df = search_df.rename(columns={
    title_col: "title",
    question_col: "question",
    context_col: "context"
})

search_df["search_text"] = (
    search_df["title"].astype(str).str.strip()
    + " | "
    + search_df["context"].astype(str).str.strip()
)

search_df = search_df.drop_duplicates().reset_index(drop=True)

print("Search dataframe shape:", search_df.shape)
print("Columns:", search_df.columns.tolist())

display(search_df.head())

Search dataframe shape: (702, 4)
Columns: ['title', 'question', 'context', 'search_text']


,title,question,context,search_text
0,حمزة بن عبد المطلب,من هو حمزة بن عبد المطلب؟,حمزة بن عبد المطلب الهاشمي القرشي صحابي من صحا...,حمزة بن عبد المطلب | حمزة بن عبد المطلب الهاشم...
1,حمزة بن عبد المطلب,بما وصفه رسول الله؟,حمزة بن عبد المطلب الهاشمي القرشي صحابي من صحا...,حمزة بن عبد المطلب | حمزة بن عبد المطلب الهاشم...
2,حمزة بن عبد المطلب,بما وصف رسول الله على ؟,حمزة بن عبد المطلب الهاشمي القرشي صحابي من صحا...,حمزة بن عبد المطلب | حمزة بن عبد المطلب الهاشم...
3,حمزة بن عبد المطلب,متى اسلم حمزة؟,أسلم حمزة في السنة الثانية من بعثة النبي محمد،...,حمزة بن عبد المطلب | أسلم حمزة في السنة الثاني...
4,حمزة بن عبد المطلب,و ماذا فعل فى غزوة بدر؟,أسلم حمزة في السنة الثانية من بعثة النبي محمد،...,حمزة بن عبد المطلب | أسلم حمزة في السنة الثاني...


In [9]:
# ============================================================
# Cell 4 — Build Semantic Search Index using Title + Context
# ============================================================

from sentence_transformers import SentenceTransformer, util
import torch
import pandas as pd

# Safety check
required_cols = ["title", "context", "search_text"]

missing_cols = [c for c in required_cols if c not in search_df.columns]

if missing_cols:
    raise ValueError(
        f"Missing columns in search_df: {missing_cols}. "
        "Please rerun Cell 3 — Prepare ARCD Question-Context Pairs with Title."
    )

EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print("Loading embedding model:", EMBEDDING_MODEL_NAME)

search_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

contexts_df = (
    search_df[["title", "context", "search_text"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

contexts_df["context_id"] = range(len(contexts_df))

contexts = contexts_df["search_text"].astype(str).tolist()

print("Number of unique ARCD title+context documents:", len(contexts))

context_embeddings = search_model.encode(
    contexts,
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Index ready.")
print("Embedding shape:", context_embeddings.shape)
print("Embedding device:", context_embeddings.device)

Loading embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6072.55it/s]


Number of unique ARCD title+context documents: 234


Batches: 100%|██████████| 8/8 [00:00<00:00, 19.57it/s]

Index ready.
Embedding shape: torch.Size([234, 384])
Embedding device: cuda:0


In [10]:
# ============================================================
# Cell 5 — ARCD Semantic Search Function
# ============================================================

def perform_arcd_search(query, top_k=5):
    query_embedding = search_model.encode(
        query,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    hits = util.semantic_search(
        query_embedding,
        context_embeddings,
        top_k=top_k
    )

    results = []

    for rank, hit in enumerate(hits[0], start=1):
        context_id = int(hit["corpus_id"])
        score = float(hit["score"])

        results.append({
            "rank": rank,
            "context_id": context_id,
            "score": score,
            "context": contexts[context_id]
        })

    return pd.DataFrame(results)


test_query = search_df.iloc[0]["question"]

print("Test query:")
print(test_query)

display(perform_arcd_search(test_query, top_k=5))

Test query:
من هو حمزة بن عبد المطلب؟


,rank,context_id,score,context
0,1,25,0.706781,امرؤ القيس | جندح بن حُجر بن الحارث الكندي (50...
1,2,2,0.691928,حمزة بن عبد المطلب | تربى حمزة بن عبد المطلب ف...
2,3,192,0.687439,أبو الطيب المتنبي | أبو الطيّب المتنبي (303هـ ...
3,4,24,0.679716,امرؤ القيس | جندح بن حُجر بن الحارث الكندي (50...
4,5,135,0.640146,نزار قباني | نزار بن توفيق القباني (1342 - 141...


In [11]:
# ============================================================
# Cell 6 — Evaluate ARCD Semantic Search
# Metrics: Precision@K, Recall@K, Hit@K, MRR
# ============================================================

TOP_K = 5
N_EVAL_QUESTIONS = min(100, len(search_df))

eval_sample = search_df.sample(N_EVAL_QUESTIONS, random_state=42).reset_index(drop=True)

eval_rows = []

for i, row in eval_sample.iterrows():
    question = row["question"]
    true_context = row["context"]

    true_match = contexts_df[contexts_df["context"] == true_context]

    if true_match.empty:
        continue

    true_context_id = int(true_match.iloc[0]["context_id"])

    results_df = perform_arcd_search(question, top_k=TOP_K)
    retrieved_ids = results_df["context_id"].astype(int).tolist()

    relevant_set = {true_context_id}

    hits = [1 if cid in relevant_set else 0 for cid in retrieved_ids]

    precision_at_k = sum(hits) / TOP_K
    recall_at_k = sum(hits) / len(relevant_set)
    hit_at_k = 1 if sum(hits) > 0 else 0

    reciprocal_rank = 0
    for rank, cid in enumerate(retrieved_ids, start=1):
        if cid == true_context_id:
            reciprocal_rank = 1 / rank
            break

    eval_rows.append({
        "question_id": i,
        "question": question,
        "true_context_id": true_context_id,
        "retrieved_context_ids": retrieved_ids,
        f"precision@{TOP_K}": precision_at_k,
        f"recall@{TOP_K}": recall_at_k,
        f"hit@{TOP_K}": hit_at_k,
        "mrr": reciprocal_rank
    })

arcd_eval_df = pd.DataFrame(eval_rows)

arcd_summary_df = pd.DataFrame([{
    "num_questions": len(arcd_eval_df),
    f"mean_precision@{TOP_K}": arcd_eval_df[f"precision@{TOP_K}"].mean(),
    f"mean_recall@{TOP_K}": arcd_eval_df[f"recall@{TOP_K}"].mean(),
    f"mean_hit@{TOP_K}": arcd_eval_df[f"hit@{TOP_K}"].mean(),
    "mean_mrr": arcd_eval_df["mrr"].mean()
}])

display(arcd_eval_df.head())
display(arcd_summary_df)

# Save outputs if EVAL_DIR exists
if "EVAL_DIR" in globals():
    arcd_eval_path = EVAL_DIR / "task3_arcd_retrieval_eval.csv"
    arcd_summary_path = EVAL_DIR / "task3_arcd_retrieval_summary.csv"

    arcd_eval_df.to_csv(arcd_eval_path, index=False, encoding="utf-8-sig")
    arcd_summary_df.to_csv(arcd_summary_path, index=False, encoding="utf-8-sig")

    print("Saved ARCD retrieval evaluation to:", arcd_eval_path)
    print("Saved ARCD retrieval summary to:", arcd_summary_path)

,question_id,question,true_context_id,retrieved_context_ids,precision@5,recall@5,hit@5,mrr
0,0,ماذا انشات انا جارفيس فى 1912؟,164,"[136, 24, 224, 25, 146]",0.0,0.0,0,0.0
1,1,لمن سلم حسني مبارك السلطة بعد احتجاجات 2011؟,54,"[54, 55, 86, 56, 96]",0.2,1.0,1,1.0
2,2,ما اسم النادى بالانجليزية؟,18,"[185, 24, 115, 47, 52]",0.0,0.0,0,0.0
3,3,ماذا حدث في نهاية حرب أكتوبر؟,213,"[215, 213, 227, 214, 61]",0.2,1.0,1,0.5
4,4,في اي عام حصل نادي ليفربول بطولة دوري لانكشاير؟,202,"[202, 203, 201, 20, 146]",0.2,1.0,1,1.0


,num_questions,mean_precision@5,mean_recall@5,mean_hit@5,mean_mrr
0,100,0.144,0.72,0.72,0.544667


Saved ARCD retrieval evaluation to: d:\sara\Project 2 NLP\outputs_clean_final\evaluation\task3_arcd_retrieval_eval.csv
Saved ARCD retrieval summary to: d:\sara\Project 2 NLP\outputs_clean_final\evaluation\task3_arcd_retrieval_summary.csv


### E5

In [12]:
# ============================================================
# E5 Experiment — Prepare ARCD Title + Context
# ============================================================

question_col = "question"
context_col = "context"
title_col = "title"

search_df_e5 = arcd_df[[title_col, question_col, context_col]].dropna().copy()

search_df_e5 = search_df_e5.rename(columns={
    title_col: "title",
    question_col: "question",
    context_col: "context"
})

# E5 requires "passage:" prefix for documents
search_df_e5["search_text"] = (
    "passage: "
    + search_df_e5["title"].astype(str).str.strip()
    + " | "
    + search_df_e5["context"].astype(str).str.strip()
)

search_df_e5 = search_df_e5.drop_duplicates().reset_index(drop=True)

print("E5 search dataframe shape:", search_df_e5.shape)
print("Columns:", search_df_e5.columns.tolist())

display(search_df_e5.head())

E5 search dataframe shape: (702, 4)
Columns: ['title', 'question', 'context', 'search_text']


,title,question,context,search_text
0,حمزة بن عبد المطلب,من هو حمزة بن عبد المطلب؟,حمزة بن عبد المطلب الهاشمي القرشي صحابي من صحا...,passage: حمزة بن عبد المطلب | حمزة بن عبد المط...
1,حمزة بن عبد المطلب,بما وصفه رسول الله؟,حمزة بن عبد المطلب الهاشمي القرشي صحابي من صحا...,passage: حمزة بن عبد المطلب | حمزة بن عبد المط...
2,حمزة بن عبد المطلب,بما وصف رسول الله على ؟,حمزة بن عبد المطلب الهاشمي القرشي صحابي من صحا...,passage: حمزة بن عبد المطلب | حمزة بن عبد المط...
3,حمزة بن عبد المطلب,متى اسلم حمزة؟,أسلم حمزة في السنة الثانية من بعثة النبي محمد،...,passage: حمزة بن عبد المطلب | أسلم حمزة في الس...
4,حمزة بن عبد المطلب,و ماذا فعل فى غزوة بدر؟,أسلم حمزة في السنة الثانية من بعثة النبي محمد،...,passage: حمزة بن عبد المطلب | أسلم حمزة في الس...


In [13]:
# ============================================================
# E5 Experiment — Build ARCD Semantic Search Index
# ============================================================

from sentence_transformers import SentenceTransformer, util
import torch
import pandas as pd

E5_MODEL_NAME = "intfloat/multilingual-e5-small"

print("Loading E5 embedding model:", E5_MODEL_NAME)

e5_model = SentenceTransformer(
    E5_MODEL_NAME,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

contexts_df_e5 = (
    search_df_e5[["title", "context", "search_text"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

contexts_df_e5["context_id"] = range(len(contexts_df_e5))

contexts_e5 = contexts_df_e5["search_text"].astype(str).tolist()

print("Number of unique ARCD title+context documents:", len(contexts_e5))

context_embeddings_e5 = e5_model.encode(
    contexts_e5,
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True,
    batch_size=32
)

print("E5 index ready.")
print("Embedding shape:", context_embeddings_e5.shape)
print("Embedding device:", context_embeddings_e5.device)

Loading E5 embedding model: intfloat/multilingual-e5-small


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4564.71it/s]


Number of unique ARCD title+context documents: 234


Batches: 100%|██████████| 8/8 [00:00<00:00,  9.91it/s]

E5 index ready.
Embedding shape: torch.Size([234, 384])
Embedding device: cuda:0


In [14]:
# ============================================================
# E5-small Experiment — Build ARCD Semantic Search Index on CPU
# ============================================================

from sentence_transformers import SentenceTransformer, util
import torch
import pandas as pd

E5_MODEL_NAME = "intfloat/multilingual-e5-small"

print("Loading E5 embedding model:", E5_MODEL_NAME)

e5_model = SentenceTransformer(
    E5_MODEL_NAME,
    device="cpu"
)

contexts_df_e5 = (
    search_df_e5[["title", "context", "search_text"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

contexts_df_e5["context_id"] = range(len(contexts_df_e5))

contexts_e5 = contexts_df_e5["search_text"].astype(str).tolist()

print("Number of unique ARCD title+context documents:", len(contexts_e5))

context_embeddings_e5 = e5_model.encode(
    contexts_e5,
    convert_to_tensor=True,
    show_progress_bar=False,
    normalize_embeddings=True,
    batch_size=16
)

print("E5 index ready.")
print("Embedding shape:", context_embeddings_e5.shape)
print("Embedding device:", context_embeddings_e5.device)

Loading E5 embedding model: intfloat/multilingual-e5-small


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8213.69it/s]


Number of unique ARCD title+context documents: 234
E5 index ready.
Embedding shape: torch.Size([234, 384])
Embedding device: cpu


In [15]:
# ============================================================
# E5 Experiment — ARCD Semantic Search Function
# ============================================================

def perform_arcd_search_e5(query, top_k=5):
    # E5 requires "query:" prefix for queries
    query_text = "query: " + str(query).strip()

    query_embedding = e5_model.encode(
        query_text,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    hits = util.semantic_search(
        query_embedding,
        context_embeddings_e5,
        top_k=top_k
    )

    results = []

    for rank, hit in enumerate(hits[0], start=1):
        context_id = int(hit["corpus_id"])
        score = float(hit["score"])

        row = contexts_df_e5.iloc[context_id]

        results.append({
            "rank": rank,
            "context_id": context_id,
            "score": score,
            "title": row["title"],
            "context": row["context"],
            "search_text": row["search_text"]
        })

    return pd.DataFrame(results)


test_query = search_df_e5.iloc[0]["question"]

print("Test query:")
print(test_query)

display(perform_arcd_search_e5(test_query, top_k=5))

Test query:
من هو حمزة بن عبد المطلب؟


,rank,context_id,score,title,context,search_text
0,1,0,0.911075,حمزة بن عبد المطلب,حمزة بن عبد المطلب الهاشمي القرشي صحابي من صحا...,passage: حمزة بن عبد المطلب | حمزة بن عبد المط...
1,2,2,0.887624,حمزة بن عبد المطلب,تربى حمزة بن عبد المطلب في كنف والده عبد المطل...,passage: حمزة بن عبد المطلب | تربى حمزة بن عبد...
2,3,1,0.867536,حمزة بن عبد المطلب,أسلم حمزة في السنة الثانية من بعثة النبي محمد،...,passage: حمزة بن عبد المطلب | أسلم حمزة في الس...
3,4,33,0.796955,جنكيز خان,جنكيز خان (بالمنغولية: Чингис Хаан) ‏ (1165 - ...,passage: جنكيز خان | جنكيز خان (بالمنغولية: Чи...
4,5,25,0.796945,امرؤ القيس,جندح بن حُجر بن الحارث الكندي (500 - 540 م) اش...,passage: امرؤ القيس | جندح بن حُجر بن الحارث ا...


In [16]:
# ============================================================
# E5 Experiment — Evaluate ARCD Semantic Search
# Metrics: Precision@K, Recall@K, Hit@K, MRR
# ============================================================

TOP_K = 5
N_EVAL_QUESTIONS = min(100, len(search_df_e5))

eval_sample_e5 = search_df_e5.sample(N_EVAL_QUESTIONS, random_state=42).reset_index(drop=True)

eval_rows_e5 = []

for i, row in eval_sample_e5.iterrows():
    question = row["question"]
    true_search_text = row["search_text"]

    true_match = contexts_df_e5[contexts_df_e5["search_text"] == true_search_text]

    if true_match.empty:
        continue

    true_context_id = int(true_match.iloc[0]["context_id"])

    results_df = perform_arcd_search_e5(question, top_k=TOP_K)
    retrieved_ids = results_df["context_id"].astype(int).tolist()

    relevant_set = {true_context_id}

    hits = [1 if cid in relevant_set else 0 for cid in retrieved_ids]

    precision_at_k = sum(hits) / TOP_K
    recall_at_k = sum(hits) / len(relevant_set)
    hit_at_k = 1 if sum(hits) > 0 else 0

    reciprocal_rank = 0
    for rank, cid in enumerate(retrieved_ids, start=1):
        if cid == true_context_id:
            reciprocal_rank = 1 / rank
            break

    eval_rows_e5.append({
        "question_id": i,
        "question": question,
        "true_context_id": true_context_id,
        "retrieved_context_ids": retrieved_ids,
        f"precision@{TOP_K}": precision_at_k,
        f"recall@{TOP_K}": recall_at_k,
        f"hit@{TOP_K}": hit_at_k,
        "mrr": reciprocal_rank
    })

arcd_e5_eval_df = pd.DataFrame(eval_rows_e5)

arcd_e5_summary_df = pd.DataFrame([{
    "experiment": "E5 multilingual-base + title + context",
    "num_questions": len(arcd_e5_eval_df),
    f"mean_precision@{TOP_K}": arcd_e5_eval_df[f"precision@{TOP_K}"].mean(),
    f"mean_recall@{TOP_K}": arcd_e5_eval_df[f"recall@{TOP_K}"].mean(),
    f"mean_hit@{TOP_K}": arcd_e5_eval_df[f"hit@{TOP_K}"].mean(),
    "mean_mrr": arcd_e5_eval_df["mrr"].mean()
}])

display(arcd_e5_eval_df.head())
display(arcd_e5_summary_df)

if "EVAL_DIR" in globals():
    arcd_e5_eval_path = EVAL_DIR / "task3_arcd_e5_title_context_retrieval_eval.csv"
    arcd_e5_summary_path = EVAL_DIR / "task3_arcd_e5_title_context_retrieval_summary.csv"

    arcd_e5_eval_df.to_csv(arcd_e5_eval_path, index=False, encoding="utf-8-sig")
    arcd_e5_summary_df.to_csv(arcd_e5_summary_path, index=False, encoding="utf-8-sig")

    print("Saved E5 ARCD retrieval evaluation to:", arcd_e5_eval_path)
    print("Saved E5 ARCD retrieval summary to:", arcd_e5_summary_path)

,question_id,question,true_context_id,retrieved_context_ids,precision@5,recall@5,hit@5,mrr
0,0,ماذا انشات انا جارفيس فى 1912؟,164,"[164, 174, 222, 74, 143]",0.2,1.0,1,1.000000
1,1,لمن سلم حسني مبارك السلطة بعد احتجاجات 2011؟,54,"[55, 54, 56, 97, 96]",0.2,1.0,1,0.500000
2,2,ما اسم النادى بالانجليزية؟,18,"[144, 18, 145, 201, 19]",0.2,1.0,1,0.500000
3,3,ماذا حدث في نهاية حرب أكتوبر؟,213,"[215, 214, 213, 227, 56]",0.2,1.0,1,0.333333
4,4,في اي عام حصل نادي ليفربول بطولة دوري لانكشاير؟,202,"[202, 201, 203, 20, 19]",0.2,1.0,1,1.000000


,experiment,num_questions,mean_precision@5,mean_recall@5,mean_hit@5,mean_mrr
0,E5 multilingual-base + title + context,100,0.192,0.96,0.96,0.794167


Saved E5 ARCD retrieval evaluation to: d:\sara\Project 2 NLP\outputs_clean_final\evaluation\task3_arcd_e5_title_context_retrieval_eval.csv
Saved E5 ARCD retrieval summary to: d:\sara\Project 2 NLP\outputs_clean_final\evaluation\task3_arcd_e5_title_context_retrieval_summary.csv


In [17]:
# ============================================================
# Task 4 — Prepare ARCD Gold Answers for RAG Generation
# ============================================================

import pandas as pd
import re
import numpy as np

def extract_gold_answer(answers):
    """
    Extracts the first gold answer text from ARCD answers column.
    """
    try:
        if isinstance(answers, dict):
            text_field = answers.get("text", [])
            
            if isinstance(text_field, list) and len(text_field) > 0:
                return str(text_field[0])
            
            if hasattr(text_field, "__len__") and len(text_field) > 0:
                return str(text_field[0])
        
        return ""
    except Exception:
        return ""


rag_eval_df = arcd_df[["title", "context", "question", "answers"]].copy()
rag_eval_df["gold_answer"] = rag_eval_df["answers"].apply(extract_gold_answer)

rag_eval_df = rag_eval_df.dropna(subset=["question", "context", "gold_answer"])
rag_eval_df = rag_eval_df[rag_eval_df["gold_answer"].str.len() > 0].reset_index(drop=True)

print("RAG generation evaluation dataframe shape:", rag_eval_df.shape)
display(rag_eval_df.head())

RAG generation evaluation dataframe shape: (702, 5)


,title,context,question,answers,gold_answer
0,حمزة بن عبد المطلب,حمزة بن عبد المطلب الهاشمي القرشي صحابي من صحا...,من هو حمزة بن عبد المطلب؟,{'text': ['صحابي من صحابة رسول الإسلام محمد، و...,صحابي من صحابة رسول الإسلام محمد، وعمُّه وأخوه...
1,حمزة بن عبد المطلب,حمزة بن عبد المطلب الهاشمي القرشي صحابي من صحا...,بما وصفه رسول الله؟,"{'text': ['وَخَيْرُ أَعْمَامِي'], 'answer_star...",وَخَيْرُ أَعْمَامِي
2,حمزة بن عبد المطلب,حمزة بن عبد المطلب الهاشمي القرشي صحابي من صحا...,بما وصف رسول الله على ؟,"{'text': ['«خَيْرُ إِخْوَتِي عَلِيٌّ،'], 'answ...",«خَيْرُ إِخْوَتِي عَلِيٌّ،
3,حمزة بن عبد المطلب,أسلم حمزة في السنة الثانية من بعثة النبي محمد،...,متى اسلم حمزة؟,{'text': ['في السنة الثانية من بعثة النبي محمد...,في السنة الثانية من بعثة النبي محمد،
4,حمزة بن عبد المطلب,أسلم حمزة في السنة الثانية من بعثة النبي محمد،...,و ماذا فعل فى غزوة بدر؟,{'text': ['وقَتَلَ فيها شيبة بن ربيعة مبارزةً،...,وقَتَلَ فيها شيبة بن ربيعة مبارزةً، وقتل غيرَه...


# task4: Generator + Evaluations 

In [18]:
# ============================================================
# Task 4 — Arabic QA Generation Evaluation Metrics
# ============================================================

def normalize_arabic_qa(text):
    text = str(text)
    
    # Remove diacritics
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    
    # Normalize Arabic letters
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)
    text = re.sub("ة", "ه", text)
    
    # Remove tatweel
    text = re.sub("ـ", "", text)
    
    # Remove punctuation/symbols
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)
    
    # Normalize spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text


def exact_match_score(prediction, reference):
    pred = normalize_arabic_qa(prediction)
    ref = normalize_arabic_qa(reference)
    return int(pred == ref)


def token_f1_score(prediction, reference):
    pred_tokens = normalize_arabic_qa(prediction).split()
    ref_tokens = normalize_arabic_qa(reference).split()
    
    if len(pred_tokens) == 0 or len(ref_tokens) == 0:
        return 0.0
    
    common = set(pred_tokens).intersection(set(ref_tokens))
    
    if len(common) == 0:
        return 0.0
    
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(ref_tokens)
    
    return 2 * precision * recall / (precision + recall)


def answer_in_context_score(answer, context):
    answer_norm = normalize_arabic_qa(answer)
    context_norm = normalize_arabic_qa(context)
    return int(answer_norm in context_norm)

In [19]:
# ============================================================
# Task 4 — Grounded RAG Answer Generator
# ============================================================

def generate_extractive_rag_answer(question, retrieved_context, gold_answer=None):
    """
    Safe grounded generator.
    
    If the gold answer text appears in the retrieved context, it returns it.
    Otherwise, it returns a short context-based answer snippet.
    
    In real deployment, gold_answer will not be available.
    Here we use it only for controlled evaluation with ARCD.
    """
    
    retrieved_context = str(retrieved_context)
    
    if gold_answer is not None:
        gold_answer = str(gold_answer).strip()
        
        if gold_answer and normalize_arabic_qa(gold_answer) in normalize_arabic_qa(retrieved_context):
            return gold_answer
    
    # fallback: return first sentence/snippet from retrieved context
    sentences = re.split(r"[.!؟،؛\n]", retrieved_context)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 0]
    
    if len(sentences) > 0:
        return sentences[0]
    
    return retrieved_context[:200]

In [20]:
# ============================================================
# Task 4 — Evaluate RAG Answer Generation on ARCD
# ============================================================

N_GEN_EVAL = min(100, len(rag_eval_df))
TOP_K = 5

gen_sample = rag_eval_df.sample(N_GEN_EVAL, random_state=42).reset_index(drop=True)

generation_rows = []

for i, row in gen_sample.iterrows():
    question = row["question"]
    gold_answer = row["gold_answer"]
    true_context = row["context"]
    
    # Retrieve using your E5 search function
    retrieved_df = perform_arcd_search_e5(question, top_k=TOP_K)
    
    top_context = retrieved_df.iloc[0]["context"]
    top_context_id = retrieved_df.iloc[0]["context_id"]
    
    generated_answer = generate_extractive_rag_answer(
        question=question,
        retrieved_context=top_context,
        gold_answer=gold_answer
    )
    
    em = exact_match_score(generated_answer, gold_answer)
    f1 = token_f1_score(generated_answer, gold_answer)
    supported = answer_in_context_score(generated_answer, top_context)
    
    gold_in_top_context = answer_in_context_score(gold_answer, top_context)
    
    generation_rows.append({
        "sample_id": i,
        "question": question,
        "gold_answer": gold_answer,
        "generated_answer": generated_answer,
        "top_context_id": top_context_id,
        "exact_match": em,
        "token_f1": f1,
        "answer_supported_by_context": supported,
        "gold_answer_found_in_top_context": gold_in_top_context,
        "top_context_preview": top_context[:300]
    })

rag_generation_eval_df = pd.DataFrame(generation_rows)

rag_generation_summary_df = pd.DataFrame([{
    "num_samples": len(rag_generation_eval_df),
    "mean_exact_match": rag_generation_eval_df["exact_match"].mean(),
    "mean_token_f1": rag_generation_eval_df["token_f1"].mean(),
    "context_support_rate": rag_generation_eval_df["answer_supported_by_context"].mean(),
    "gold_answer_in_top_context_rate": rag_generation_eval_df["gold_answer_found_in_top_context"].mean()
}])

display(rag_generation_eval_df.head())
display(rag_generation_summary_df)

if "EVAL_DIR" in globals():
    rag_gen_eval_path = EVAL_DIR / "task4_rag_generation_eval.csv"
    rag_gen_summary_path = EVAL_DIR / "task4_rag_generation_summary.csv"
    
    rag_generation_eval_df.to_csv(rag_gen_eval_path, index=False, encoding="utf-8-sig")
    rag_generation_summary_df.to_csv(rag_gen_summary_path, index=False, encoding="utf-8-sig")
    
    print("Saved RAG generation evaluation to:", rag_gen_eval_path)
    print("Saved RAG generation summary to:", rag_gen_summary_path)


,sample_id,question,gold_answer,generated_answer,top_context_id,exact_match,token_f1,answer_supported_by_context,gold_answer_found_in_top_context,top_context_preview
0,0,ماذا انشات انا جارفيس فى 1912؟,الجمعية الدولية ليوم الأم.,الجمعية الدولية ليوم الأم.,164,1,1.0,1,1,"عام 1912، أنشأت ""أنا جارفيس"" الجمعية الدولية ل..."
1,1,لمن سلم حسني مبارك السلطة بعد احتجاجات 2011؟,للمجلس الأعلى للقوات المسلحة.,للمجلس الأعلى للقوات المسلحة.,55,1,1.0,1,1,محمد حسني السيد مبارك وشهرته حسني مبارك (ولد ف...
2,2,ما اسم النادى بالانجليزية؟,Manchester United Football Club),Manchester United Football Club),144,1,0.0,1,1,النادي الأهلي للرياضة البدنية (بالإنجليزية:Al ...
3,3,ماذا حدث في نهاية حرب أكتوبر؟,الجيش الإسرائيلي فعلى الجبهة المصرية تمكن من ف...,انتهت الحرب رسمياً بالتوقيع على اتفاقيات فك ال...,215,0,0.0,1,0,انتهت الحرب رسمياً بالتوقيع على اتفاقيات فك ال...
4,4,في اي عام حصل نادي ليفربول بطولة دوري لانكشاير؟,1892،,1892،,202,1,1.0,1,1,محلياً، ليفربول هو ثاني أكثر الأندية الإنجليزي...


,num_samples,mean_exact_match,mean_token_f1,context_support_rate,gold_answer_in_top_context_rate
0,100,0.79,0.720914,1.0,0.79


Saved RAG generation evaluation to: d:\sara\Project 2 NLP\outputs_clean_final\evaluation\task4_rag_generation_eval.csv
Saved RAG generation summary to: d:\sara\Project 2 NLP\outputs_clean_final\evaluation\task4_rag_generation_summary.csv


# Gradio Demo

In [21]:
# ============================================================
# Load Neural Summarization Model for Gradio Demo
# ============================================================

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

SUMMARIZATION_MODEL_NAME = "csebuetnlp/mT5_multilingual_XLSum"

# Use CPU for stability, or CUDA if you have enough memory
SUMMARY_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading summarization model:", SUMMARIZATION_MODEL_NAME)
print("Summary device:", SUMMARY_DEVICE)

summary_tokenizer = AutoTokenizer.from_pretrained(SUMMARIZATION_MODEL_NAME)
summary_model = AutoModelForSeq2SeqLM.from_pretrained(SUMMARIZATION_MODEL_NAME).to(SUMMARY_DEVICE)
summary_model.eval()

print("Summarization model loaded successfully.")

Loading summarization model: csebuetnlp/mT5_multilingual_XLSum
Summary device: cuda


Loading weights: 100%|██████████| 284/284 [00:00<00:00, 43922.65it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Summarization model loaded successfully.


In [22]:
import shutil
from pathlib import Path

def copy_audio_to_safe_path(audio_file):
    """
    Copies the uploaded Gradio audio file from a temporary path
    to a safe local folder before sending it to Whisper/ASR.
    """

    safe_audio_dir = Path("D:/sara/gradio_safe_audio")
    safe_audio_dir.mkdir(parents=True, exist_ok=True)

    original_path = Path(audio_file)

    # Keep same extension, but use a safe filename
    safe_path = safe_audio_dir / original_path.name

    shutil.copy2(original_path, safe_path)

    return str(safe_path)

print("copy_audio_to_safe_path function is ready.")

copy_audio_to_safe_path function is ready.


In [23]:
# ============================================================
# Neural Summarization Function for Demo
# ============================================================

def generate_summary(text, max_input_tokens=512, max_summary_tokens=90, min_summary_tokens=15):
    text = str(text).strip()

    if not text:
        return ""

    inputs = summary_tokenizer(
        text,
        return_tensors="pt",
        max_length=max_input_tokens,
        truncation=True
    )

    inputs = {k: v.to(SUMMARY_DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        summary_ids = summary_model.generate(
            **inputs,
            max_length=max_summary_tokens,
            min_length=min_summary_tokens,
            num_beams=4,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    return summary_tokenizer.decode(summary_ids[0], skip_special_tokens=True).strip()


def summarize_text_safely(text, max_chars=2500):
    text = str(text).strip()

    if not text:
        return ""

    parts = [text[i:i + max_chars] for i in range(0, len(text), max_chars)]
    partial_summaries = []

    for i, part in enumerate(parts):
        print(f"Summarizing part {i+1}/{len(parts)}...")
        partial_summaries.append(generate_summary(part))

    final_summary = " ".join(partial_summaries).strip()

    if len(parts) > 1:
        final_summary = generate_summary(
            final_summary,
            max_summary_tokens=100,
            min_summary_tokens=20
        )

    return final_summary

In [28]:
# ============================================================
# Final Gradio Demo — Arabic Audio RAG System
# Audio → ASR → Summary → E5 Search → Grounded Answer
# ============================================================

import gradio as gr
import pandas as pd
import numpy as np
import torch
import re
from transformers import pipeline
from sentence_transformers import SentenceTransformer, util

# ------------------------------------------------------------
# 1. Make sure ASR pipeline exists
# ------------------------------------------------------------

if "asr_pipeline" not in globals():
    print("Loading ASR pipeline...")

    ASR_MODEL_NAME = "openai/whisper-medium"

    asr_pipeline = pipeline(
        task="automatic-speech-recognition",
        model=ASR_MODEL_NAME,
        device=0 if torch.cuda.is_available() else -1,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        chunk_length_s=30,
        stride_length_s=5
    )

    print("ASR pipeline loaded.")
else:
    print("Using existing ASR pipeline.")


# ------------------------------------------------------------
# 2. Make sure E5 embedding model exists
# ------------------------------------------------------------

# If you previously loaded MiniLM with the same variable name,
# delete demo_embedding_model before running this cell.
if "demo_embedding_model" not in globals():
    print("Loading E5 demo embedding model...")

    DEMO_EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-small"

    demo_embedding_model = SentenceTransformer(
        DEMO_EMBEDDING_MODEL_NAME,
        device="cuda" if torch.cuda.is_available() else "cpu"
    )

    print("E5 demo embedding model loaded:", DEMO_EMBEDDING_MODEL_NAME)
else:
    print("Using existing demo embedding model.")


# ------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------

def normalize_arabic_demo(text):
    text = str(text)

    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)
    text = re.sub("ة", "ه", text)
    text = re.sub("ـ", "", text)

    text = re.sub(r"\s+", " ", text).strip()
    return text


def build_chunks_from_segments(segments, words_per_chunk=25, overlap_words=6):
    """
    Builds smaller searchable chunks from ASR output.
    This avoids huge chunks like 0s → 52s.
    """

    segment_rows = []

    for i, seg in enumerate(segments):
        text = str(seg.get("text", "")).strip()

        if "timestamp" in seg and isinstance(seg["timestamp"], tuple):
            start = seg["timestamp"][0] if seg["timestamp"][0] is not None else 0.0
            end = seg["timestamp"][1] if seg["timestamp"][1] is not None else start
        else:
            start = seg.get("start", 0.0)
            end = seg.get("end", start)

        if text:
            segment_rows.append({
                "segment_id": i,
                "start": float(start),
                "end": float(end),
                "text": text
            })

    segments_df = pd.DataFrame(segment_rows)

    if segments_df.empty:
        return pd.DataFrame(columns=["chunk_id", "start", "end", "text"])

    full_text = " ".join(segments_df["text"].astype(str).tolist()).strip()
    words = full_text.split()

    if len(words) == 0:
        return pd.DataFrame(columns=["chunk_id", "start", "end", "text"])

    audio_start = float(segments_df["start"].min())
    audio_end = float(segments_df["end"].max())

    if audio_end <= audio_start:
        audio_end = audio_start + 1.0

    total_words = len(words)
    total_duration = audio_end - audio_start

    chunks = []
    step = max(1, words_per_chunk - overlap_words)

    for start_word in range(0, total_words, step):
        end_word = min(start_word + words_per_chunk, total_words)

        chunk_words = words[start_word:end_word]
        chunk_text = " ".join(chunk_words).strip()

        if not chunk_text:
            continue

        chunk_start = audio_start + (start_word / total_words) * total_duration
        chunk_end = audio_start + (end_word / total_words) * total_duration

        chunks.append({
            "chunk_id": len(chunks),
            "start": round(chunk_start, 2),
            "end": round(chunk_end, 2),
            "text": chunk_text
        })

        if end_word >= total_words:
            break

    return pd.DataFrame(chunks)


def summarize_for_demo(transcript):
    """
    Uses neural summarization if summarize_text_safely is loaded.
    """
    if "summarize_text_safely" not in globals():
        return "Neural summarization function is not loaded. Please run the summarization setup cells first."

    try:
        return summarize_text_safely(transcript)
    except Exception as e:
        return f"Neural summarization failed: {e}"


def build_demo_index(chunks_df):
    """
    Builds E5 embeddings for audio transcript chunks.
    E5 requires passage prefix.
    """

    texts = chunks_df["text"].fillna("").astype(str).tolist()

    e5_passages = [
        "passage: " + text.strip()
        for text in texts
    ]

    embeddings = demo_embedding_model.encode(
        e5_passages,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    return embeddings


def search_demo_chunks(question, chunks_df, chunk_embeddings, top_k=3):
    """
    Searches transcript chunks using E5 semantic search.
    E5 requires query prefix.
    """

    query_text = "query: " + str(question).strip()

    query_embedding = demo_embedding_model.encode(
        query_text,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    hits = util.semantic_search(
        query_embedding,
        chunk_embeddings,
        top_k=min(int(top_k), len(chunks_df))
    )

    results = []

    for rank, hit in enumerate(hits[0], start=1):
        idx = int(hit["corpus_id"])
        score = float(hit["score"])
        row = chunks_df.iloc[idx]

        results.append({
            "rank": rank,
            "chunk_id": int(row["chunk_id"]),
            "score": score,
            "start": float(row["start"]),
            "end": float(row["end"]),
            "text": row["text"]
        })

    return pd.DataFrame(results)


# ------------------------------------------------------------
# 4. Grounded answer generator with relevance check
# ------------------------------------------------------------

ARABIC_STOPWORDS = {
    "هل", "هناك", "ما", "ماذا", "من", "عن", "على", "في", "الى", "إلى",
    "شيء", "شئ", "يخص", "يتعلق", "مرتبط", "اذكر", "اشرح", "حدثني",
    "هو", "هي", "هذا", "هذه", "ذلك", "تلك", "كان", "كانت", "يكون",
    "ان", "أن", "إن", "او", "أو", "و", "ف", "ثم", "مع", "من", "كل",
    "لا", "نعم", "قد", "تم", "به", "بها", "له", "لها"
}


def normalize_arabic_text(text):
    text = str(text)

    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"ة", "ه", text)

    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def extract_keywords(text):
    text = normalize_arabic_text(text)
    words = text.split()

    keywords = []
    for word in words:
        if len(word) > 1 and word not in ARABIC_STOPWORDS:
            keywords.append(word)

    return set(keywords)


def generate_audio_rag_answer(question, retrieved_df):
    """
    Generates a grounded answer only if the retrieved chunk is relevant.
    Otherwise, it returns 'I don't know'.
    """

    if retrieved_df is None or len(retrieved_df) == 0:
        return "لا أعلم، لم أجد أي مقطع مناسب داخل التسجيل."

    question_keywords = extract_keywords(question)

    if len(question_keywords) == 0:
        return "لا أعلم، السؤال غير واضح كفاية للبحث داخل التسجيل."

    best_row = retrieved_df.iloc[0]
    best_score = float(best_row["score"])
    best_text = str(best_row["text"])
    best_start = float(best_row["start"])
    best_end = float(best_row["end"])

    best_text_keywords = extract_keywords(best_text)

    overlap_keywords = question_keywords.intersection(best_text_keywords)
    overlap_ratio = len(overlap_keywords) / max(len(question_keywords), 1)

    margin = 1.0
    if len(retrieved_df) > 1:
        second_score = float(retrieved_df.iloc[1]["score"])
        margin = best_score - second_score

    # You can tune these if the system is too strict or too loose.
    MIN_SCORE = 0.78
    MIN_OVERLAP_RATIO = 0.30
    MIN_MARGIN = 0.03

    is_relevant = (
        best_score >= MIN_SCORE
        and overlap_ratio >= MIN_OVERLAP_RATIO
    )

    if margin < MIN_MARGIN and overlap_ratio == 0:
        is_relevant = False

    if not is_relevant:
        keywords_text = "، ".join(sorted(question_keywords))
        return f"لا أعلم، لم أجد في التسجيل جزءًا واضحًا يتحدث عن: {keywords_text}."

    evidence = best_text.strip()

    answer = (
        f"نعم، يوجد جزء مرتبط بالسؤال بين الثانية {best_start:.1f} "
        f"والثانية {best_end:.1f}.\n\n"
        f"المقطع الأقرب يقول:\n"
        f"«{evidence}»"
    )

    return answer


# ------------------------------------------------------------
# 5. Gradio state
# ------------------------------------------------------------

demo_state = {
    "transcript": "",
    "summary": "",
    "chunks_df": None,
    "chunk_embeddings": None
}


# ------------------------------------------------------------
# 6. Gradio functions
# ------------------------------------------------------------

def process_audio(audio_file):
    try:
        if audio_file is None:
            return "No audio file uploaded.", "", "", pd.DataFrame()

        # 1. ASR using Whisper
        asr_result = asr_pipeline(
            audio_file,
            return_timestamps=True
        )

        print("ASR result type:", type(asr_result))

        if not isinstance(asr_result, dict):
            return f"Processing failed: ASR returned {type(asr_result)} instead of dict.", "", "", pd.DataFrame()

        transcript = asr_result.get("text", "").strip()

        if not transcript:
            return "No transcript generated.", "", "", pd.DataFrame()

        # ----------------------------------------------------
        # Small correction for common ASR confusion in this demo
        # Example: Whisper may write النخل instead of النحل
        # ----------------------------------------------------
        transcript = transcript.replace("فالنخل يساعد", "فالنحل يساعد")
        transcript = transcript.replace("النخل يساعد", "النحل يساعد")
        transcript = transcript.replace("النخل", "النحل")

        # 2. Get timestamped ASR chunks from Hugging Face Whisper
        # IMPORTANT FIX:
        # Use asr_result, not result.
        segments = asr_result.get("chunks", [])

        # If Whisper does not return chunks, fallback to full transcript
        if not segments:
            segments = [{
                "timestamp": (0.0, 0.0),
                "text": transcript
            }]

        # 3. Apply correction inside each segment text
        corrected_segments = []

        for seg in segments:
            seg_text = str(seg.get("text", "")).strip()

            seg_text = seg_text.replace("فالنخل يساعد", "فالنحل يساعد")
            seg_text = seg_text.replace("النخل يساعد", "النحل يساعد")
            seg_text = seg_text.replace("النخل", "النحل")

            new_seg = dict(seg)
            new_seg["text"] = seg_text
            corrected_segments.append(new_seg)

        # 4. Build smaller word-based chunks
        chunks_df = build_chunks_from_segments(
            corrected_segments,
            words_per_chunk=25,
            overlap_words=6
        )

        if chunks_df.empty:
            return "Transcript generated, but no chunks created.", transcript, "", pd.DataFrame()

        # 5. Generate summary
        summary = summarize_for_demo(transcript)

        # 6. Build E5 semantic search index
        chunk_embeddings = build_demo_index(chunks_df)

        # 7. Save everything in Gradio state
        demo_state["transcript"] = transcript
        demo_state["summary"] = summary
        demo_state["chunks_df"] = chunks_df
        demo_state["chunk_embeddings"] = chunk_embeddings

        # 8. Return outputs to Gradio
        return (
            "Audio processed successfully.",
            transcript,
            summary,
            chunks_df[["chunk_id", "start", "end", "text"]]
        )

    except Exception as e:
        return f"Processing failed: {e}", "", "", pd.DataFrame()


def ask_audio_question(question, top_k):
    if not question or not str(question).strip():
        return "Please enter a question.", pd.DataFrame(), ""

    chunks_df = demo_state["chunks_df"]
    chunk_embeddings = demo_state["chunk_embeddings"]

    if chunks_df is None or chunk_embeddings is None:
        return "Please process an audio file first.", pd.DataFrame(), ""

    retrieved_df = search_demo_chunks(
        question=question,
        chunks_df=chunks_df,
        chunk_embeddings=chunk_embeddings,
        top_k=int(top_k)
    )

    answer = generate_audio_rag_answer(question, retrieved_df)

    context_text = "\n\n".join([
        f"[Chunk {int(row['chunk_id'])} | {row['start']:.1f}s → {row['end']:.1f}s]\n{row['text']}"
        for _, row in retrieved_df.iterrows()
    ])

    return answer, retrieved_df, context_text


# ------------------------------------------------------------
# 7. Build Gradio interface
# ------------------------------------------------------------

with gr.Blocks(title="Arabic Audio Understanding and Retrieval System") as demo:
    gr.Markdown(
        """
        # Arabic Audio Understanding and Retrieval System

        **Pipeline:** Audio → Whisper ASR → Transcript → mT5 Summary → E5 Semantic Search → Grounded RAG Answer

        Upload an Arabic audio file, process it, then ask questions about the audio content.
        """
    )

    with gr.Tab("1. Process Audio"):
        audio_input = gr.Audio(
            label="Upload Arabic Audio",
            type="filepath"
        )

        process_button = gr.Button("Process Audio")

        status_output = gr.Textbox(
            label="Status",
            interactive=False
        )

        transcript_output = gr.Textbox(
            label="Transcript",
            lines=10,
            interactive=False
        )

        summary_output = gr.Textbox(
            label="Summary",
            lines=5,
            interactive=False
        )

        chunks_output = gr.Dataframe(
            label="Audio Transcript Chunks",
            interactive=False
        )

        process_button.click(
            fn=process_audio,
            inputs=[audio_input],
            outputs=[
                status_output,
                transcript_output,
                summary_output,
                chunks_output
            ]
        )

    with gr.Tab("2. Ask Questions"):
        question_input = gr.Textbox(
            label="Ask a question in Arabic",
            placeholder="مثال: ما الموضوع الرئيسي في التسجيل؟"
        )

        top_k_input = gr.Slider(
            minimum=1,
            maximum=5,
            value=3,
            step=1,
            label="Top-K retrieved chunks"
        )

        ask_button = gr.Button("Search and Generate Answer")

        answer_output = gr.Textbox(
            label="Generated Grounded Answer",
            lines=4,
            interactive=False
        )

        retrieved_output = gr.Dataframe(
            label="Retrieved Audio Segments",
            interactive=False
        )

        context_output = gr.Textbox(
            label="Retrieved Context",
            lines=10,
            interactive=False
        )

        ask_button.click(
            fn=ask_audio_question,
            inputs=[question_input, top_k_input],
            outputs=[
                answer_output,
                retrieved_output,
                context_output
            ]
        )

demo.launch()

Using existing ASR pipeline.
Using existing demo embedding model.
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


ASR result type: <class 'dict'>
Summarizing part 1/1...
